# Boltz-2 Batch — GFET Probe Structures
Corre o Boltz-2 em todas as probes do ZIP gerado pelo pipeline.  
Requer GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
# Célula 1 — Instalar Boltz-2
!pip install boltz -q
print("Boltz instalado.")

In [ ]:
# Célula 2 — Upload do ZIP e extracção dos YAMLs
import zipfile
from pathlib import Path
from google.colab import files

print("Faz upload do ficheiro boltz2_inputs.zip ...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

yaml_dir = Path("yamls")
yaml_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(yaml_dir)

yaml_files = sorted(yaml_dir.glob("*.yaml"))
print(f"{len(yaml_files)} probes encontradas:")
for y in yaml_files:
    print(f"  {y.stem}")

In [ ]:
# Célula 3 — Correr Boltz-2 em todas as probes
import subprocess, json, csv

out_root = Path("boltz_results")
out_root.mkdir(exist_ok=True)
results = []

for i, yf in enumerate(yaml_files, 1):
    probe_id = yf.stem
    out_dir  = out_root / probe_id
    print(f"[{i}/{len(yaml_files)}] {probe_id}", flush=True)

    subprocess.run(
        ["boltz", "predict", str(yf),
         "--out_dir",          str(out_dir),
         "--recycling_steps",  "3",
         "--sampling_steps",   "200",
         "--diffusion_samples","1",
         "--accelerator",      "gpu",
         "--model",            "boltz2"],
        capture_output=True, text=True
    )

    conf_files = list(out_dir.glob("**/confidence_*.json")) or list(out_dir.glob("**/*.json"))
    confidence = ptm = plddt = None
    if conf_files:
        d = json.load(open(conf_files[0]))
        confidence = d.get("confidence_score") or d.get("confidence")
        ptm        = d.get("ptm")
        raw        = d.get("plddt")
        plddt      = round(sum(raw)/len(raw), 3) if isinstance(raw, list) else (d.get("plddt_score") or d.get("mean_plddt"))

    cif_files = list(out_dir.glob("**/*.cif"))
    status    = "OK" if cif_files else "FAILED"
    results.append({
        "probe_id":   probe_id,
        "status":     status,
        "confidence": round(confidence, 3) if confidence else "",
        "ptm":        round(ptm, 3)        if ptm        else "",
        "plddt":      round(plddt, 3)      if plddt      else "",
        "cif_path":   str(cif_files[0])    if cif_files  else "",
    })
    c = f"{confidence:.3f}" if confidence else "N/A"
    p = f"{ptm:.3f}"        if ptm        else "N/A"
    l = f"{plddt:.3f}"      if plddt      else "N/A"
    print(f"  {chr(10003) if status=="OK" else chr(10007)}  confidence={c}  pTM={p}  pLDDT={l}")

print(f"
Concluido: {sum(1 for r in results if r["status"]=="OK")}/{len(results)} com sucesso.")

In [ ]:
# Célula 4 — Resumo dos resultados
import pandas as pd

df = pd.DataFrame(results)

def quality(row):
    c = row["confidence"]
    if c == "": return "N/A"
    c = float(c)
    if c >= 0.80: return "HIGH"
    if c >= 0.60: return "MODERATE"
    return "LOW"

df["quality"] = df.apply(quality, axis=1)
df.to_csv("boltz2_results_summary.csv", index=False)
print(df[["probe_id","status","confidence","ptm","plddt","quality"]].to_string(index=False))
print("
CSV guardado: boltz2_results_summary.csv")

In [ ]:
# Célula 5 — Download dos resultados
import shutil
shutil.make_archive("boltz2_all_results", "zip", str(out_root))
files.download("boltz2_all_results.zip")
files.download("boltz2_results_summary.csv")
print("Download iniciado.")